# Exploring 311 dataset for department resource utilization

In [ ]:
# Importing all the necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import random
import matplotlib as mpl
import seaborn as sns
%matplotlib inline
plt.style.use(['fivethirtyeight'])
mpl.rcParams['lines.linewidth'] = 3

## Pre-processing 

As part of pre-processing, we've followed the following techniques:
1) Data Sampling 
2) Data Cleaning
3) Data Integration
4) Data Reduction
5) Feature Engineering

### Data Sampling

Because of the huge size of the dataset, we've used random sampling to create a smaller dataset containing similar distributions 
as in our primary dataset. We've created a method **GenerateDataSample()** to generate a smaller dataset and store it as '311_sampled_data.csv'

In [ ]:
def GenerateDataSample():
    # Importing the necessary libraries
    import pandas as pd
    import random

    # Specifying the CSV file path
    csv_file = '311_Service_Requests.csv'

    # Read a random sample of 1% of the data into the dataframe
    df = pd.read_csv(csv_file, skiprows=lambda i: i > 0 and random.random() > 0.01)

    # Creating the random data sample dataset
    df.to_csv('311_sampled_data.csv', index=False)

    return df

In [ ]:
# Calling the function to generate the sample dataset
df = GenerateDataSample()
df.head()

### Data Cleaning

The sample data also contains lot of inconsistent and null value data, and needs to be transformed accordingly by either filling with appropriate values / deleting the rows and more. In this section, we clean the 311 dataset and the zip code dataset, which would be later used for analysis and modelling. We've created different methods to execute the tasks, out of which **Clean311Dataset()** and **CleanZipCodeDataset()** are the primary methods for cleaning our datasets.
The two datasets used are
1) 311 sampled dataset
2) Zip code dataset

In [ ]:
import pandas as pd
def DoCleaning():
    # Cleaning the 311 dataset
    df311 = Clean311Dataset()
    # Cleaning the zipcode dataset
    dfZip = CleanZipCodeDataset()

    return df311, dfZip

def Clean311Dataset():
    """ To clean the 311 dataset """
    # Using the existing the dataset from the variable if provided
    df = pd.read_csv('311_sampled_data.csv')
    
    # To delete all the columns with majority NaN values
    df = DeleteNaNColumns(df)

    # To filter the rows which do not have any close date
    df = df[df['Closed Date'].notna()]

    # The descriptor contains values like Partial Access, Window Frame, Door, Other/Unknown 
    # We impute the Descriptor values => Other/Unknown for the ones where they are NaN.
    df['Descriptor'] = df['Descriptor'].fillna('Other/Unknown') 

    # We fill the NaN Landmark values as 'Unspecified'
    df['Landmark'] = df['Landmark'].fillna('Unspecified')
    
    # Populating the value for NaN records as 'Address Unknown' because it is already
    # present as an existing label in the column.
    df['Location Type'] = df['Location Type'].fillna('Address Unknown')  

    # For NaN city values, we populate it with Boroughs
    df['City'] = df.City.fillna(df['Borough'])

    # Removing NaN boroughs    
    df = df[df['Borough'].notna()]

    # Removing NaN city values
    df = df[df['City'].notna()]

    # Removing NaN incident addresses
    df = df[df['Incident Address'].notna()]

    # Removing NaN street names
    df = df[df['Street Name'].notna()]
    
    # To remove the nan address types from the dataset.
    df = df[df['Address Type'].notna()]
    
    # Filtering the rows with NaN values for 'Incident Zip' column.
    # We filter this because, this column will be used later for joining with other tables
    df = df[df['Incident Zip'].notna()]

    # Filtering the rows with NaN values for 'Latitude' and 'Longitude' columns.
    # These records are being filtered as to identify the locations for the complaints
    df = df[df['Latitude'].notna()]

    # To filter NaN BBL values from the dataset
    df = df[df['BBL'].notna()]

    # To filter out the rows based on departments.
    df = FilterDepartments(df)

    #df.to_csv('311_clean_data_1.csv',index=False)
    return df

def DeleteNaNColumns(df):
    """ To delete all the columns which have majority (more than 90%) of NaN values """
    nan_columns_to_be_deleted = []
    for col in df.columns:
        isna_count = df[col].isna().sum()
        notna_count = df[col].notna().sum()
        if (notna_count <= 0.1 * (len(df))):
            nan_columns_to_be_deleted.append(col)
    print ("The columns with 90% NaN values in our dataset (would be removed) :")
    print (nan_columns_to_be_deleted)
    df = df.drop(columns = nan_columns_to_be_deleted)
    return df

def isNaColumns(df):
    """ To provide summary of na and non-na values for all the columns """
    for col in df_sample.columns:
        isna_count = df_sample[col].isna().sum()
        notna_count = df_sample[col].notna().sum()
        print (col + " => IsNA : " + str(isna_count) + " Not NA : " + str(notna_count))

def FilterDepartments(df):
    """ To filter the departments in the dataset """
    PlotParetoChart(df)
    # Based on Pareto's principal, we found that 80% of the data is represented by the
    # top 5 departments. 
    # So, we would be filtering our departments.
    # Note: But in our study, we observed that filtering out the rows per department 
    # are creating an issue of loss in some features, and for that reason we filter out at a later stage.
    df_temp = df['Agency'].value_counts()
    return df

def PlotParetoChart(df):
    """ Plotting the Pareto chart for the complaint department wise """
    # Retrieving the frequency of complaints department-wise
    df_new = df['Agency'].value_counts()
    
    # Calculating the cumulative sum of the complaints department-wise
    departments = ( df_new.cumsum() / df_new.sum() ) * 100

    # Plotting the figures
    print ("Here is the Pareto's chart")
    fig, ax = plt.subplots()
    ax.barh(df_new.index, df_new.values, color="C0")
    ax.set_xlabel("Complaint frequency")
    ax.set_ylabel("Departments")
    ax2 = ax.twiny()
    ax2.plot(departments, df_new.index, color="C1", marker="D", ms=7)
    ax2.set_xlim([0, 100])
    ax2.set_xlabel('Cumulative percentage')
    
    plt.show()

def CleanZipCodeDataset():
    """ To clean the zip code dataset """
    import geopandas as gpd
    gdf = gpd.read_file('ZIP_CODE_040114.shp')

    # Excluding the records with 0 population
    gdf = gdf[gdf['POPULATION'] > 0.0]

    # Excluding the records with area <= 0 
    gdf = gdf[gdf['AREA'] > 0.0 ]

    print (gdf.head(10))
    #gdf.to_csv('zip_code_dataset.csv', index=False)
    return gdf
    # Note: Other columns are irrelevant and would not be cleaned as part of the cleaning phase.
    # Those columns would be removed from the dataset in the reduction phase. 


In [ ]:
df311, dfZip = DoCleaning()

In [ ]:
df311

### Data Integration

To augment our existing 311 dataset by including more features for analysis and modelling, we're using the **Zip Code** dataset. We merge both the datasets on **ZipCode** attribute and leverage the features from the new dataset like *Population*, *Area* and *geometry of boroughs*.
We have **IntegrateDatasets()** method to handle the merging of the dataset.

In [ ]:
import pandas as pd
import numpy as np

def IntegrateDatasets(df311, dfZip):
    """ To integrate 311 and zip code datasets """
    df_311 = Preprocess311Dataset(df311)
    df_zip = PreprocessZipCodeDataset(dfZip)
    
    # Merging the datasets
    merged_df = df_311.merge(df_zip, on='ZIPCODE')

    #merged_df.to_csv('311_integrated_data.csv',index=False)
    return merged_df

def Preprocess311Dataset(df_temp):
    """ To load the 311 dataset and do the necessary preparations for the merge """
    
    # Rename the 'Incident Zip' column to 'Zipcode' for merging
    df_temp.rename(columns = {'Incident Zip':'ZIPCODE'}, inplace=True)
    df_temp['ZIPCODE'] = df_temp['ZIPCODE'].astype('int16')
    
    return df_temp

def PreprocessZipCodeDataset(dfZip):
    """ To preprocess the ZIPCODE column of the dataset """
    # Casting to int
    dfZip['ZIPCODE'] = dfZip['ZIPCODE'].astype('int16')
    return dfZip    


In [ ]:
df_int = IntegrateDatasets(df311, dfZip)

In [ ]:
# Displaying the merged dataset
df_int

### Data Reduction

During our analysis, we noticed that there were some features were redundant and some of them irrelevant to objective of our analysis as well as for the modelling. In this section, we drop all these features and is taken care by the **DoReduction()** method in this section.

In [ ]:
import pandas as pd

def DoReduction(df_merged):
    """ To remove unnecessary columns from the dataset """
    df_merged = DropDuplicateFeatures(df_merged)
    df_merged = DropIrrelevantFeatures(df_merged)

    #df_merged.to_csv('311_reduced_data.csv',index=False)
    return df_merged

def DropDuplicateFeatures(df):
    """ To drop duplicate features """
    duplicate_cols = ['Agency Name','COUNTY','CTY_FIPS','PO_NAME','Location']
    return df.drop(columns = duplicate_cols)

def DropIrrelevantFeatures(df):
    """ To drop irrelevant features """
    irrelevant_cols = ['Resolution Description','Resolution Action Updated Date','Status','ZIPCODE','Incident Address','Street Name',
                       'BLDGZIP','geometry','STATE','URL','SHAPE_AREA','SHAPE_LEN','ST_FIPS','Cross Street 1','Cross Street 2',
                       'Intersection Street 1','Intersection Street 2','X Coordinate (State Plane)','Y Coordinate (State Plane)',
                       'Park Facility Name','Facility Type','Park Borough','Due Date','Unique Key','Descriptor','Landmark','AREA']
    return df.drop(columns = irrelevant_cols)


In [ ]:
df_int = DoReduction(df_int)

In [ ]:
df_int

### Feature Engineering

Our target variable(s) for classification and regression models constituted derivation based on existing attributes in the dataset. 
1) For the regression model, we wanted to predict the *Resolution Time* for the complaints, which we derived from *Closed Date* and *Created Date* attributes of our dataset.
2) For the classification model, we created a new class label *Additional Resources Required*, which was derived from the *Resolution Time* attribute by considering department-wise the average resolution time, and labeled it as 1 if the *Resolution Time* was greater than the average, else we label it as 0.

Apart from the above, we also split the *Created Date* attribute into day, month, year and weekday respectively and drop the parent attribute (*Created Date*)

Note: The feature engineering for classification can be found as part of the **Modelling** notebook.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

def DoFeatureEngineering(df_FE):
    """ To do feature engineering and create target variables for regression and classification models """
    # Creating Resolution Time variable for the dataset (Target Variable)
    df_FE = CreateResolutionTimeFeature(df_FE)

    # Splitting the created date feature to grab patterns
    # in days, months or years
    df_FE = SplitCreatedDateFeature(df_FE)

    cols_to_be_dropped = ['Closed Date','Created Date']
    df_FE = df_FE.drop(columns = cols_to_be_dropped)

    return df_FE

def CreateResolutionTimeFeature(df):
    """ To create Resolution Time target variable """
    df['Created Date'] = pd.to_datetime(df['Created Date'])
    df['Closed Date'] = pd.to_datetime(df['Closed Date'])
    df['Resolution Time'] = df['Closed Date'] - df['Created Date']

    df = df[df['Resolution Time'] > pd.Timedelta(0)]

    return df

def SplitCreatedDateFeature(df):
    """ Splitting the Created Date feature """
    df['Created_Date_Year'] = df['Created Date'].dt.year
    df['Created_Date_Month'] = df['Created Date'].dt.month
    df['Created_Date_Day'] = df['Created Date'].dt.day
  
    return df



In [ ]:
# We perform feature engineering by calling the function 
df_int = DoFeatureEngineering(df_int)

In [ ]:
df_int

In [ ]:
df_int.to_csv('311_preprocessed_dataset.csv',index=False)